# 16 — API Contracts, Versioning, Pagination, and Evolution

Good APIs remain understandable and compatible as implementations evolve. Explore explicit schemas, pagination tokens, error responses, versioning rules, and backward-compatibility trade-offs using offline data.

**Level:** Advanced Python · **Study time:** 90–150 minutes · **Prerequisites:** Basics and Intermediate; **Environment:** Python 3.11+ (validated on 3.12), standard library.

## Learning outcomes
- Specify API request/response contracts
- Handle pagination and bounded traversal
- Design stable structured error responses
- Plan compatible schema evolution

**How to work:** Predict, execute, vary inputs, and test the failure path. Examples are offline and intentionally small; discuss how they would evolve with production traffic.

## Theory, demonstration, and reasoning

Each section contains a runnable code cell and a question requiring you to justify design choices.

### 1. API contracts and validation

A well-defined response includes stable fields and documented types. New optional fields are generally easier to add than silently repurposing existing fields. A consumer must validate external input, even if the server claims a schema.

In [ ]:
def normalize_response(raw):
    if not isinstance(raw, dict) or not isinstance(raw.get("items"), list):
        raise ValueError("items list required")
    version = raw.get("version", 1)
    if version not in (1, 2):
        raise ValueError("unsupported version")
    return {"version": version, "items": list(raw["items"])}

print(normalize_response({"items": [1, 2], "version": 2}))

**Review:** Which response changes would break an older client that expects a number?

### 2. Error envelopes

An error contract should distinguish stable machine-readable codes from human-friendly descriptions. Avoid leaking internal stack traces and secrets through public responses.

In [ ]:
def error_response(code, message, request_id):
    return {
        "error": {
            "code": code,
            "message": message,
            "request_id": request_id,
        }
    }

response = error_response("INVALID_ARGUMENT", "page_size must be positive", "request-01")
print(response["error"]["code"])

**Review:** Why should clients branch on stable codes rather than literal English messages?

### 3. Pagination and loop safeguards

Cursor pagination can be preferable when records are inserted concurrently, but every traversal needs a termination rule and maximum page count. Treat cursor tokens as opaque rather than trying to decode their implementation.

In [ ]:
PAGES = {
    "start": {"items": ["A", "B"], "next": "two"},
    "two": {"items": ["C"], "next": None},
}
def collect(fetch, cursor="start", max_pages=10):
    visited, collected = set(), []
    while cursor is not None:
        if cursor in visited or len(visited) >= max_pages:
            raise RuntimeError("loop or page limit")
        visited.add(cursor)
        page = fetch(cursor)
        collected.extend(page["items"])
        cursor = page.get("next")
    return collected

print(collect(PAGES.__getitem__))

**Review:** What does cursor pagination protect against that a simple offset may not?

### 4. Versioned serializers

Maintain stable old output when introducing new optional fields. Use explicit transformations rather than relying on dumping an object's complete __dict__, which could unintentionally disclose internal data.

In [ ]:
def serialize_product(product, version=1):
    common = {"sku": product["sku"], "name": product["name"]}
    if version == 1:
        return common
    if version == 2:
        return {**common, "category": product.get("category")}
    raise ValueError("unsupported API version")

record = {"sku": "P1", "name": "Pen", "category": "stationery"}
print(serialize_product(record, 1), serialize_product(record, 2))

**Review:** When is it preferable to release a new API version rather than add an optional field?

## Independent challenges

Define contracts and error behavior before coding. Test normal, empty, malformed and boundary values where applicable.

**Exercise 1.** Design a paginated catalog contract and document error responses.

In [ ]:
# Exercise 1
# TODO: implement and test.

**Exercise 2.** Write a versioned serializer supporting v1 and v2 without changing old output.

In [ ]:
# Exercise 2
# TODO: implement and test.

**Exercise 3.** Test a client that encounters a repeated next-page cursor.

In [ ]:
# Exercise 3
# TODO: implement and test.

**Exercise 4.** Explain compatibility risks from changing a field from string to number.

In [ ]:
# Exercise 4
# TODO: implement and test.

## Pitfalls and readiness

- Never serialize arbitrary internal objects into public API responses.
- Use bounded pagination and validate cursor policies.
- Versioning does not excuse missing input validation.

**Reflection:** What must be logged and measured? What must never be exposed? What assumptions fail under concurrency or large input?

Complete the corresponding Advanced exercise pack and rerun this notebook from a fresh kernel.